In [ ]:
#dataset
import pandas as pd
import random

topics = {
    "education": [
        "Students should study regularly to perform well in exams",
        "Education helps people gain knowledge and skills",
        "Teachers guide students toward academic success",
        "Learning new subjects improves critical thinking"
    ],

    "sports": [
        "Athletes train daily to improve performance",
        "The football team practiced hard before the match",
        "Regular exercise helps build stamina",
        "Fans celebrated the team's victory"
    ],

    "health": [
        "Eating healthy food improves overall well being",
        "Doctors recommend regular physical activity",
        "Drinking water is essential for the human body",
        "Sleep is necessary for physical recovery"
    ],

    "travel": [
        "Traveling allows people to experience new cultures",
        "Tourists enjoy visiting historical landmarks",
        "Families often plan vacations during holidays",
        "Exploring new cities can be exciting"
    ],

    "food": [
        "Cooking at home can be healthy and enjoyable",
        "Restaurants serve many traditional dishes",
        "Street food markets attract many visitors",
        "Different cultures have unique cuisines"
    ],

    "environment": [
        "Planting trees helps protect the environment",
        "Climate change affects ecosystems globally",
        "Recycling reduces environmental pollution",
        "Clean energy reduces carbon emissions"
    ],

    "business": [
        "Entrepreneurs start businesses to create opportunities",
        "Marketing strategies help increase sales",
        "Good management improves productivity",
        "Companies focus on satisfying customer needs"
    ],

    "daily_life": [
        "People enjoy spending time with family",
        "Reading books increases knowledge",
        "Listening to music can improve mood",
        "Morning walks improve mental health"
    ],

    "technology": [
        "Technology changes how people communicate",
        "Smartphones are widely used in modern life",
        "Software development requires logical thinking",
        "Cloud computing allows remote data storage"
    ]
}

paraphrase_templates = [
    "In other words, {}",
    "This means that {}",
    "Simply put, {}",
    "It can also be said that {}",
]

rows = []
target = 50000

for i in range(target):

    topic = random.choice(list(topics.keys()))
    sentence1 = random.choice(topics[topic])

    if random.random() < 0.5:
        # paraphrase case
        sentence2 = random.choice(paraphrase_templates).format(sentence1.lower())
        label = 1
    else:
        # non paraphrase
        other_topic = random.choice(list(topics.keys()))
        sentence2 = random.choice(topics[other_topic])
        label = 0

    rows.append([sentence1, sentence2, label])

df = pd.DataFrame(rows, columns=["text1","text2","label"])

df.to_csv("paraphrase.csv", index=False)

print("Dataset generated:", len(df))

Dataset generated: 50000


In [ ]:
# Paraphrase Detection Model
# SBERT + Cosine Similarity + Logistic Regression

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sentence_transformers import SentenceTransformer


# -----------------------------
# 1 Load Dataset
# -----------------------------

print("Loading dataset...")

df = pd.read_csv("paraphrase.csv")

print("Dataset shape:", df.shape)
print(df.head())


# -----------------------------
# 2 Train Test Split
# -----------------------------

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))


# -----------------------------
# 3 Load SBERT Model
# -----------------------------

print("Loading SBERT model...")

model = SentenceTransformer("all-mpnet-base-v2")


# -----------------------------
# 4 Generate Embeddings
# -----------------------------

print("Generating embeddings...")

train_text1_embeddings = model.encode(
    train_df["text1"].tolist(),
    show_progress_bar=True
)

train_text2_embeddings = model.encode(
    train_df["text2"].tolist(),
    show_progress_bar=True
)

test_text1_embeddings = model.encode(
    test_df["text1"].tolist(),
    show_progress_bar=True
)

test_text2_embeddings = model.encode(
    test_df["text2"].tolist(),
    show_progress_bar=True
)


# -----------------------------
# 5 Compute Cosine Similarity
# -----------------------------

print("Computing similarity scores...")

train_similarity = []

for i in range(len(train_text1_embeddings)):

    sim = cosine_similarity(
        [train_text1_embeddings[i]],
        [train_text2_embeddings[i]]
    )[0][0]

    train_similarity.append(sim)

train_similarity = np.array(train_similarity).reshape(-1,1)


test_similarity = []

for i in range(len(test_text1_embeddings)):

    sim = cosine_similarity(
        [test_text1_embeddings[i]],
        [test_text2_embeddings[i]]
    )[0][0]

    test_similarity.append(sim)

test_similarity = np.array(test_similarity).reshape(-1,1)


# -----------------------------
# 6 Train Logistic Regression
# -----------------------------

print("Training classifier...")

classifier = LogisticRegression()

classifier.fit(
    train_similarity,
    train_df["label"]
)


# -----------------------------
# 7 Predictions
# -----------------------------

predictions = classifier.predict(test_similarity)


# -----------------------------
# 8 Model Evaluation
# -----------------------------

accuracy = accuracy_score(
    test_df["label"],
    predictions
)

precision = precision_score(
    test_df["label"],
    predictions
)

recall = recall_score(
    test_df["label"],
    predictions
)

f1 = f1_score(
    test_df["label"],
    predictions
)

print("\nModel Evaluation Results\n")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)


# -----------------------------
# 9 Confusion Matrix
# -----------------------------

matrix = confusion_matrix(
    test_df["label"],
    predictions
)

print("\nConfusion Matrix:\n")
print(matrix)


# -----------------------------
# 10 Classification Report
# -----------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        test_df["label"],
        predictions
    )
)


# -----------------------------
# 11 Paraphrase Detection Function
# -----------------------------

def detect_paraphrase(text1, text2):

    emb1 = model.encode([text1])
    emb2 = model.encode([text2])

    similarity = cosine_similarity(emb1, emb2)[0][0]

    prediction = classifier.predict([[similarity]])[0]

    if prediction == 1:
        result = "Paraphrase"
    else:
        result = "Not Paraphrase"

    return similarity, result


# -----------------------------
# 12 Example Test
# -----------------------------

print("\nExample Test...\n")

text_a = "Machine learning improves automation in industries"
text_b = "Automation in industries is improved by machine learning"

score, result = detect_paraphrase(text_a, text_b)

print("Text 1:", text_a)
print("Text 2:", text_b)

print("\nSimilarity Score:", score)
print("Prediction:", result)


# -----------------------------
# 13 User Input Test
# -----------------------------

if __name__ == "__main__":

    print("\n--- Paraphrase Detection Manual Test ---")

    while True:

        text1 = input("\nEnter Text 1:\n")
        text2 = input("\nEnter Text 2:\n")

        score, result = detect_paraphrase(text1, text2)

        print("\nSimilarity Score:", score)
        print("Prediction:", result)

        again = input("\nCheck another pair? (y/n): ")

        if again.lower() != "y":
            print("\nProgram Ended.")
            break

Loading dataset...
Dataset shape: (50000, 3)
                                            text1  \
0       Technology changes how people communicate   
1  Drinking water is essential for the human body   
2               Reading books increases knowledge   
3    Planting trees helps protect the environment   
4      Cloud computing allows remote data storage   

                                               text2  label  
0  This means that technology changes how people ...      1  
1  In other words, drinking water is essential fo...      1  
2      Simply put, reading books increases knowledge      1  
3  The football team practiced hard before the match      0  
4               Regular exercise helps build stamina      0  
Training samples: 40000
Testing samples: 10000
Loading SBERT model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings...


Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Computing similarity scores...
Training classifier...

Model Evaluation Results

Accuracy : 0.9826
Precision: 0.966787554876885
Recall   : 1.0
F1 Score : 0.9831133540372671

Confusion Matrix:

[[4761  174]
 [   0 5065]]

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.96      0.98      4935
           1       0.97      1.00      0.98      5065

    accuracy                           0.98     10000
   macro avg       0.98      0.98      0.98     10000
weighted avg       0.98      0.98      0.98     10000


Example Test...

Text 1: Machine learning improves automation in industries
Text 2: Automation in industries is improved by machine learning

Similarity Score: 0.9250624
Prediction: Paraphrase

--- Paraphrase Detection Manual Test ---

Enter Text 2:
Artificial intelligence is transforming modern industries.

Similarity Score: 0.63595164
Prediction: Not Paraphrase


In [ ]:
# Paraphrase Detection Model
# SBERT + Cosine Similarity + Logistic Regression

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sentence_transformers import SentenceTransformer


# -----------------------------
# 1 Load Dataset
# -----------------------------

print("Loading dataset...")

df = pd.read_csv("paraphrase.csv")

print("Dataset shape:", df.shape)
print(df.head())


# -----------------------------
# 2 Train Test Split
# -----------------------------

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))


# -----------------------------
# 3 Load SBERT Model
# -----------------------------

print("Loading SBERT model...")

model = SentenceTransformer("all-mpnet-base-v2")


# -----------------------------
# 4 Generate Embeddings
# -----------------------------

print("Generating embeddings...")

train_text1_embeddings = model.encode(
    train_df["text1"].tolist(),
    show_progress_bar=True
)

train_text2_embeddings = model.encode(
    train_df["text2"].tolist(),
    show_progress_bar=True
)

test_text1_embeddings = model.encode(
    test_df["text1"].tolist(),
    show_progress_bar=True
)

test_text2_embeddings = model.encode(
    test_df["text2"].tolist(),
    show_progress_bar=True
)


# -----------------------------
# 5 Compute Cosine Similarity
# -----------------------------

print("Computing similarity scores...")

train_similarity = []

for i in range(len(train_text1_embeddings)):

    sim = cosine_similarity(
        [train_text1_embeddings[i]],
        [train_text2_embeddings[i]]
    )[0][0]

    train_similarity.append(sim)

train_similarity = np.array(train_similarity).reshape(-1,1)


test_similarity = []

for i in range(len(test_text1_embeddings)):

    sim = cosine_similarity(
        [test_text1_embeddings[i]],
        [test_text2_embeddings[i]]
    )[0][0]

    test_similarity.append(sim)

test_similarity = np.array(test_similarity).reshape(-1,1)


# -----------------------------
# 6 Train Logistic Regression
# -----------------------------

print("Training classifier...")

classifier = LogisticRegression()

classifier.fit(
    train_similarity,
    train_df["label"]
)


# -----------------------------
# 7 Predictions
# -----------------------------

predictions = classifier.predict(test_similarity)


# -----------------------------
# 8 Model Evaluation
# -----------------------------

accuracy = accuracy_score(
    test_df["label"],
    predictions
)

precision = precision_score(
    test_df["label"],
    predictions
)

recall = recall_score(
    test_df["label"],
    predictions
)

f1 = f1_score(
    test_df["label"],
    predictions
)

print("\nModel Evaluation Results\n")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)


# -----------------------------
# 9 Confusion Matrix
# -----------------------------

matrix = confusion_matrix(
    test_df["label"],
    predictions
)

print("\nConfusion Matrix:\n")
print(matrix)


# -----------------------------
# 10 Classification Report
# -----------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        test_df["label"],
        predictions
    )
)


# -----------------------------
# 11 Paraphrase Detection Function
# -----------------------------

def detect_paraphrase(text1, text2):

    emb1 = model.encode([text1])
    emb2 = model.encode([text2])

    similarity = cosine_similarity(emb1, emb2)[0][0]

    prediction = classifier.predict([[similarity]])[0]

    if prediction == 1:
        result = "Paraphrase"
    else:
        result = "Not Paraphrase"

    return similarity, result


# -----------------------------
# 12 Example Test
# -----------------------------

print("\nExample Test...\n")

text_a = "Machine learning improves automation in industries"
text_b = "Automation in industries is improved by machine learning"

score, result = detect_paraphrase(text_a, text_b)

print("Text 1:", text_a)
print("Text 2:", text_b)

print("\nSimilarity Score:", score)
print("Prediction:", result)


# -----------------------------
# 13 User Input Test
# -----------------------------

if __name__ == "__main__":

    print("\n--- Paraphrase Detection Manual Test ---")

    while True:

        text1 = input("\nEnter Text 1:\n")
        text2 = input("\nEnter Text 2:\n")

        score, result = detect_paraphrase(text1, text2)

        print("\nSimilarity Score:", score)
        print("Prediction:", result)

        again = input("\nCheck another pair? (y/n): ")

        if again.lower() != "y":
            print("\nProgram Ended.")
            break